# Spark in a Databricks notebook


## 1. Load the table

`spark.table` takes a three-part Unity Catalog name: catalog, schema, table. This reads `sample_dataset` from the `default` schema of the `workspace` catalog.

| Column | Type | Meaning |
| --- | --- | --- |
| `full_name` | string | Synthetic person name |
| `email` | string | Synthetic email |
| `city` | string | Synthetic city |
| `job` | string | Job title |
| `registered_on` | date | Registration date |
| `salary` | int | Annual salary (28,000–220,000) |
| `years_of_experience` | int | Years of experience (0–40) |
| `gender` | string | Female, Male, or Non-binary |
| `latitude`, `longitude` | double | Synthetic coordinates |

In [ ]:
TABLE = "workspace.default.sample_dataset"

people_raw = spark.table(TABLE)
print(f"{people_raw.count():,} rows from {TABLE}")
people_raw.printSchema()
display(people_raw.limit(10))

## Describe the people table

`describe()` prints **count**, **mean**, **stddev**, **min**, and **max**. `gender` is a string, so the useful rows there are count, min, and max, and those min and max values are alphabetical.

`summary()` adds the quartiles. The **50%** row is the median salary and the median years of experience. The median sits at the middle of the values, which is the figure to read when a few high salaries pull the mean up.

In [ ]:
display(people_raw.select("salary", "years_of_experience", "gender").describe())
display(people_raw.select("salary", "years_of_experience").summary())

## 2. Reshape the people data

The summaries above describe the raw table. This section adds the registration year, an experience band, and a salary band, then lists senior high earners and pay by experience band.

The reshaped frame is registered as the temporary view `people`. The SQL in the next sections reads that view.

In [ ]:
from pyspark.sql import functions as F

people = (
    people_raw
    .withColumn("registered_year", F.year("registered_on"))
    .withColumn("registered_month", F.month("registered_on"))
    .withColumn(
        "experience_band",
        F.when(F.col("years_of_experience") < 5, "0-4 years")
        .when(F.col("years_of_experience") < 10, "5-9 years")
        .when(F.col("years_of_experience") < 20, "10-19 years")
        .otherwise("20+ years"),
    )
    .withColumn(
        "band_order",
        F.when(F.col("years_of_experience") < 5, 1)
        .when(F.col("years_of_experience") < 10, 2)
        .when(F.col("years_of_experience") < 20, 3)
        .otherwise(4),
    )
    .withColumn(
        "salary_band",
        F.when(F.col("salary") < 60000, "Under 60k")
        .when(F.col("salary") < 100000, "60k-99k")
        .when(F.col("salary") < 150000, "100k-149k")
        .otherwise("150k+"),
    )
)


people.createOrReplaceTempView("people")
print(f"{people.count():,} people cached as view 'people'")

seniors = (
    people
    .filter((F.col("years_of_experience") >= 20) & (F.col("salary") >= 150000))
    .select(
        "full_name",
        "job",
        "city",
        "years_of_experience",
        "salary",
        "registered_on",
    )
    .orderBy(F.desc("salary"))
)
print(f"Senior high earners (20+ years and salary >= 150k): {seniors.count():,}")
display(seniors.limit(15))

In [ ]:
by_experience = (
    people.groupBy("band_order", "experience_band")
    .agg(
        F.count("*").alias("people"),
        F.round(F.avg("salary"), 0).alias("avg_salary"),
        F.expr("percentile_approx(salary, 0.5)").alias("median_salary"),
        F.min("salary").alias("min_salary"),
        F.max("salary").alias("max_salary"),
    )
    .orderBy("band_order")
    .drop("band_order")
)
display(by_experience)

by_gender = (
    people.groupBy("gender")
    .agg(
        F.count("*").alias("people"),
        F.round(F.avg("salary"), 0).alias("avg_salary"),
        F.expr("percentile_approx(salary, 0.5)").alias("median_salary"),
    )
    .orderBy(F.desc("people"))
)
display(by_gender)

## 3. Simple Spark SQL on the people table

Same questions as the DataFrame aggregations above, written in SQL against the `people` view. After the year query, click **+** above the result and choose **Visualization** if you want a bar chart of registrations by year.

In [ ]:
display(spark.sql("""
    SELECT
        gender,
        COUNT(*) AS people,
        ROUND(AVG(salary), 0) AS avg_salary,
        percentile_approx(salary, 0.5) AS median_salary
    FROM people
    GROUP BY gender
    ORDER BY people DESC
"""))

In [ ]:
display(spark.sql("""
    SELECT
        registered_year,
        COUNT(*) AS registrations,
        ROUND(AVG(salary), 0) AS avg_salary
    FROM people
    GROUP BY registered_year
    ORDER BY registered_year
"""))

## 4. More queries on the people table

These stay on one table. `LIMIT` keeps the first rows after the sort. The city query uses `WHERE` before `GROUP BY`, so it counts only people whose salary is at least 100,000.

In [ ]:
display(spark.sql("""
    SELECT
        job,
        COUNT(*) AS people,
        ROUND(AVG(salary), 0) AS avg_salary,
        MIN(salary) AS min_salary,
        MAX(salary) AS max_salary
    FROM people
    GROUP BY job
    ORDER BY avg_salary DESC, job
    LIMIT 10
"""))

In [ ]:
display(spark.sql("""
    SELECT
        experience_band,
        COUNT(*) AS people,
        ROUND(AVG(salary), 0) AS avg_salary
    FROM people
    GROUP BY experience_band
    ORDER BY MIN(years_of_experience)
"""))

In [ ]:
display(spark.sql("""
    SELECT
        city,
        COUNT(*) AS people,
        ROUND(AVG(salary), 0) AS avg_salary
    FROM people
    WHERE salary >= 100000
    GROUP BY city
    ORDER BY people DESC, city
    LIMIT 10
"""))